# Step 2 — Spark through Livy: guided activities

Use this notebook as a **Markdown-only learning guide**. All analysis code is
supplied in `02_spark_via_livy.py`. Read the explanations here, then copy only the
current authorised command into the prepared **JupyterLab terminal**. Do not
execute the displayed Python snippets in a notebook kernel.

For every activity: **read → predict → copy the command → inspect the output →
record and explain your findings → continue**. Double-click a "Your answers"
cell, edit it, render it, and save your notebook.

Complete Step 1 and resolve its cleanup first. Step 2 uses the same supplied
bank-loan CSV through a different submission path; it is not a new dataset,
predictive model, or scaling benchmark.

**Pilot status: guided participant execution remains unvalidated.** The original
organiser-owned full-analysis tests do not validate the new activity commands,
participant access, concurrent capacity, or failure-path recovery. Source tests
and file publication are not proof of readiness. Wait for separate Step 2 approval.

### Learning objectives

- Trace the HTTP client, Livy-side driver, master, worker and executors.
- Explain session creation, readiness polling, statement IDs and returned results.
- Compare schema inspection, selection, filtering and aggregation with Step 1.
- Explain bounded JSON result transfer and participant-local CSV writing.
- Distinguish an owned-session deletion request, HTTP 404 confirmation, and
  independent verification of Spark application shutdown.


## Understand what automation has prepared

The organiser supplies the assigned Livy service, Spark runtime, dataset mounts,
Python environment and protected access. You do not install software, deploy
containers, expose services, invent settings, or manage Docker during this exercise.

### Where execution happens

1. Your **JupyterLab terminal** runs the Python HTTP client.
2. With no explicit Livy overrides, the access helper uses your group's managed
   SSH identity to create temporary **loopback-only forwarding** to the assigned
   private Livy service. Requests do not go to a guessed public endpoint.
3. **Livy** creates a PySpark session and receives Python statements through HTTP.
   The Spark **driver runs on the Livy side**, not in your notebook kernel or the
   Step 1 direct-submission client.
4. The **master** coordinates resource allocation; the assigned VM's **worker**
   launches **executors** that process data partitions.
5. Statement output returns through HTTP. An exporting activity returns bounded
   rows as JSON, which the client writes to a CSV in the participant workspace.

The pilot is VM-local: one assigned worker per VM, not a Spark cluster spanning
all workshop VMs. A small CSV and grouped results do not demonstrate scaling.

### Managed access and dataset checks

The existing helper reads `~/.ssh/spark-direct/login_key` and `known_hosts`.
The key must be a participant-owned regular file with private permissions; the
host record pins exactly one assigned endpoint. Do not display or exchange keys,
replace host pins, make manual tunnels, or create alternative access directories.

With neither `LIVY_URL` nor `LIVY_DATA_PATH` supplied, the helper verifies the local
CSV's SHA-256 against bytes read on the Livy driver and **one executor**. This
checks the supplied snapshot and that executor's access, not multi-worker capacity.
The local path and the driver/executor path need not be identical.

Complete explicit organiser overrides remain supported; partial overrides fail.
In the explicit path, this automatic dataset verification is **not** performed,
so the organiser must validate that path. Participants must not set overrides.
JupyterHub login alone does not secure Livy's API; do not expose the service or
weaken HTTP/TLS, SSH, routing, or host-key safeguards.

### Reflect before submitting

- How does the driver location differ from Step 1?
- What does SSH forwarding provide, and what does Livy provide?
- Why check CSV bytes on both the driver and an executor?
- What does this verification leave untested?


### Your answers

**Driver, client and executor locations compared with Step 1**

_Write your answer here._

**Forwarding, dataset verification and their limitations**

_Write your answer here._

**What automation prepared and what participants must not change**

_Write your answer here._


## Before starting: files, sessions and completion

Locate this notebook, `02_spark_via_livy.py`, `spark_livy_access.py`, and
`data/Bank_Personal_Loan_Modelling.csv`. Use matching updated copies. Reuse the
README and Step 1 discussion of provenance, permissions, field definitions and
negative `Experience` values. Do not modify the supplied CSV to match a count.

In the prepared JupyterLab terminal, copy only:

```bash
cd ~/Big-Data/Spark
```

This directory change submits no job. Wait for organiser permission before an
activity. **One active submission per group; take turns.** The client has no
concurrency lock. Fresh session IDs do not establish concurrent capacity.

### Session and statement readiness

Each activity command creates **one new owned Livy session**, using the existing
`{'kind': 'pyspark'}` creation payload. This guided change adds no resource limits;
do not assume Step 1's explicit limits apply. The organiser must approve the
existing session configuration and launch path.

The client uses the returned session ID, waits until the session is idle, then
submits statements. Each statement also returns an ID, which the client polls
until output is available. Statement IDs are handled by the client, not chosen
by participants. Within that one session, later statements can use variables
created by earlier statements. **No DataFrame survives into another command.**

Record the printed session ID, Spark version and master URL. IDs are scoped to
the assigned Livy service; never assume zero or reuse another group's ID.
A returned ID is not proof of readiness, and a successful local CSV preview is
not proof of remote access. Every command repeats preparation and, in managed
mode, dataset verification; that verification itself performs Spark processing.

The existing client uses 15-second HTTP request timeouts, 120-second session and
statement wait windows, and 2-second polling intervals. These are not one hard
time limit for the whole command. Owned-session removal has a separate 60-second
verification window; individual requests can extend polling beyond a deadline.

### Completion checks for every activity

Check the selected analysis output, then:

- `Owned-session removal confirmed; cluster containers remain running.`
- `Analysis completed successfully; temporary managed forwarding has closed.`
- A zero process exit status; immediately after the activity, copy `echo $?`.

In explicit-override mode there is no temporary forwarding, although the existing
final message still mentions it. Activities 1–4 should not print `Saved:`.
Do not continue after an error, interruption, or unconfirmed cleanup.

### Predict

Why must the client wait for both session readiness and statement completion?
What can happen if session creation succeeds remotely but its response is lost?


### Your answers

**Session readiness, statement readiness and ID ownership**

_Write your answer here._

**Lost creation responses and why retrying can be unsafe**

_Write your answer here._


## How the five activities work

| Activity | Analysis task | Writes the Step 2 preview? |
|---|---|---|
| 1 | Load CSV, inspect schema and count input | No |
| 2 | Select columns | No |
| 3 | Filter rows with Age > 30 | No |
| 4 | Count raw Personal Loan categories | No |
| 5 | Recreate the filter and export a bounded preview | Yes |

Activity numbers identify tasks, **not notebook cell positions**. Each command
owns a separate session, performs its required preparation, runs only the selected
analysis, and attempts owned-session removal. Activity 5 does not depend on
activity 3's old variables. Repeated schema inference can itself require work.

For each activity, record predictions first, run one command, inspect its result
and completion, then answer the reflection questions. Do not paste all commands
as a batch. Existing previews are not evidence of success for activities 1–4.

The Python illustrations below are taken from the supplied client. The string
`PREPARED_LIVY_DATA_PATH` is an illustrative placeholder: the helper supplies the
actual reviewed path automatically. **Do not run the snippets, edit the path, or
export settings yourself.** Only copy the activity's terminal command.


## Activity 1 — Load the CSV and inspect its schema

**Purpose:** Understand how CSV bytes become a DataFrame with named, inferred types. Distinguish the ordinary participant-local CSV preview from the remote Spark display.

### Predict

Which types do you expect for Age, Income and Family? Does the CSV declare a schema? Why does showing five records not establish the input count?

### Read the supplied code — do not execute it locally

```python
df = (spark.read.format('csv')
      .option('header', 'true')
      .option('inferSchema', 'true')
      .load('PREPARED_LIVY_DATA_PATH'))
df.printSchema()
df.show(5)
print('Number of records:', df.count())
```

### Copy to the terminal

```bash
python3 02_spark_via_livy.py --activity 1
```

### Observe and reflect

Record the local preview versus remote display, inferred types, full input count, session ID, Spark version/master, and completion checks. Keep relevant excerpts rather than all logs.

- What do header and inferSchema do?
- Compare the schema and count with Step 1 for the same snapshot.
- Why does a local preview not prove executor data access?
- Which field meanings require documentation rather than inference from numeric values?


### Your answers

**Predictions before execution**

_Write your answer here._

**Observed results, session ID, exit status and completion checks**

_Write your answer here._

**Explanation and comparison with Step 1**

_Write your answer here._


## Activity 2 — Select columns

**Purpose:** Distinguish changing columns from changing records. select builds a transformation plan; show and DataFrame count request results and are actions.

### Predict

Which columns will remain? Should column selection change the number of records compared with activity 1? Explain.

### Read the supplied code — do not execute it locally

```python
df = (spark.read.format('csv')
      .option('header', 'true')
      .option('inferSchema', 'true')
      .load('PREPARED_LIVY_DATA_PATH'))

selected_df = df.select('Age', 'Income', 'Family')
selected_df.show(10)
print('Selected columns:', ', '.join(selected_df.columns))
print('Records after column selection:', selected_df.count())
```

### Copy to the terminal

```bash
python3 02_spark_via_livy.py --activity 2
```

### Observe and reflect

Record the selected columns, full selected-record count and completion checks. Distinguish the ten displayed records from the total count.

- Did the result match your prediction?
- Which lines build a plan and which request execution?
- How does this new session obtain its DataFrame?
- What stays the same as Step 1 despite the different submission path?


### Your answers

**Predictions before execution**

_Write your answer here._

**Observed results, session ID, exit status and completion checks**

_Write your answer here._

**Explanation and comparison with Step 1**

_Write your answer here._


## Activity 3 — Filter rows using Age > 30

**Purpose:** Understand a strict condition and distinguish row filtering from column selection. The activity selects Age, Income and Family, then filters matching records.

### Predict

Does age exactly 30 match? Should the matching count be greater than, equal to, or less than the full input count?

### Read the supplied code — do not execute it locally

```python
df = (spark.read.format('csv')
      .option('header', 'true')
      .option('inferSchema', 'true')
      .load('PREPARED_LIVY_DATA_PATH'))

result_df = df.select('Age', 'Income', 'Family').filter(df.Age > 30)
result_df.show(10)
print('Matching records:', result_df.count())
```

### Copy to the terminal

```bash
python3 02_spark_via_livy.py --activity 3
```

### Observe and reflect

Record the matching count, columns, a small displayed excerpt and completion checks. Compare the count with Step 1; report differences rather than changing the CSV.

- Why does age exactly 30 not match?
- Which operations build a plan and which request results?
- Does the display contain all matching records?
- Without explicit ordering, should reruns necessarily display the same first ten rows?


### Your answers

**Predictions before execution**

_Write your answer here._

**Observed results, session ID, exit status and completion checks**

_Write your answer here._

**Explanation and comparison with Step 1**

_Write your answer here._


## Activity 4 — Count Personal Loan categories

**Purpose:** Interpret aggregation over the full input, not the age-filtered result. Raw category codes are retained; consult source documentation before assigning meanings.

### Predict

What should the grouped counts sum to? Does sorting the summary by category value change the counts?

### Read the supplied code — do not execute it locally

```python
df = (spark.read.format('csv')
      .option('header', 'true')
      .option('inferSchema', 'true')
      .load('PREPARED_LIVY_DATA_PATH'))

df.groupBy('Personal Loan').count().orderBy('Personal Loan').show()
```

### Copy to the terminal

```bash
python3 02_spark_via_livy.py --activity 4
```

### Observe and reflect

Record the categories and counts, their sum, and completion checks. Compare with activity 1 and Step 1 for the supplied snapshot.

- Why is comparing the sum with the full input count useful?
- Which category is more frequent, and what may the documentation justify saying about its meaning?
- Why could unequal frequencies matter in later classification?
- Why do grouped results not prove multi-worker execution or causation?


### Your answers

**Predictions before execution**

_Write your answer here._

**Observed results, session ID, exit status and completion checks**

_Write your answer here._

**Explanation and comparison with Step 1**

_Write your answer here._


## Activity 5 — Export and retrieve a bounded preview

**Purpose:** Distinguish Livy-side collection, bounded HTTP result transfer and participant-local CSV writing. The activity recreates its filter, then collects at most ten rows.

### Predict

Will the file contain every matching record? Where are rows collected, how do they travel back, and where is the CSV finally written?

### Read the supplied code — do not execute it locally

```python
df = (spark.read.format('csv')
      .option('header', 'true')
      .option('inferSchema', 'true')
      .load('PREPARED_LIVY_DATA_PATH'))

result_df = df.select('Age', 'Income', 'Family').filter(df.Age > 30)

def export_preview(base_url: str, session_id: int, output_dir: Path) -> Path:
    """Transfer at most ten rows as JSON and write a participant-local CSV.

    Unlike Exercise 1, collection happens at the Livy-side driver. The bounded
    result travels back over HTTP. No shared worker file is written or changed.
    """
    code = "import json\nprint(" + repr(PREVIEW_MARKER) + " + json.dumps({" \
        "'columns': result_df.columns, " \
        "'rows': [list(row) for row in result_df.limit(10).collect()]}))"
    text = run_statement(base_url, session_id, code)
    lines = [line for line in text.splitlines() if line.startswith(PREVIEW_MARKER)]
    if len(lines) != 1:
        raise RuntimeError("Livy did not return exactly one bounded preview payload.")
    preview = json.loads(lines[0][len(PREVIEW_MARKER):])
    columns, rows = preview["columns"], preview["rows"]
    if columns != ["Age", "Income", "Family"] or len(rows) > 10:
        raise ValueError("The returned preview does not match the expected columns or row limit.")
    if any(not isinstance(row, list) or len(row) != len(columns) for row in rows):
        raise ValueError("The returned preview contains an invalid row.")
    output_dir.mkdir(exist_ok=True)
    output_file = output_dir / "02_filtered_preview.csv"
    with output_file.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(columns)
        writer.writerows(rows)
    return output_file
```

### Copy to the terminal

```bash
python3 02_spark_via_livy.py --activity 5
```

### Observe and reflect

Check Saved: separately from session cleanup and final success. Open the printed path in JupyterLab when using the authorised participant-terminal path. Record the Age,Income,Family header and number of data rows, excluding the header.

- How does this JSON-over-HTTP result path differ from Step 1's SFTP download?
- Why is collecting ten records safer than collecting an arbitrarily large result?
- How does the single CSV differ from Spark's distributed CSV output directory?
- Why does a result file alone not prove that owned-session cleanup succeeded?


### Result location and reruns

The client writes `outputs/02_filtered_preview.csv` under its own workspace
path after validating the returned JSON columns and row limit. An empty filter
produces a header-only file. It writes no shared remote preview file.

An exporting rerun replaces this Step 2 CSV. Retain a needed result before
rerunning. It does not replace Step 1's `outputs/remote-<unique-id>/` downloads or
the source dataset. Do not remove the ten-row limit. If an organiser-approved
wrapper instead runs inside a client container, its printed path may require
separate retrieval and is not automatically visible in JupyterLab.

A result-transfer or write failure is not permission to resubmit. Report the
failure and cleanup status; an older local file may still exist.


### Your answers

**Predictions before execution**

_Write your answer here._

**Observed results, session ID, exit status and completion checks**

_Write your answer here._

**Explanation and comparison with Step 1**

_Write your answer here._


## Finish, distinguish cleanup evidence and report uncertainty

After obtaining a valid returned session ID, the client attempts removal of
**that owned session only**, even after an analysis error. It then polls the
same service and ID until GET returns **HTTP 404** before reporting removal.
An accepted DELETE alone is insufficient.

HTTP 404 confirms that the owned Livy session resource is no longer available
through that API. It does **not** independently prove that all Spark applications
or executors have terminated. The organiser's application/worker checks remain
separate. Shared runtime containers remain running.

A timeout, lost response, disconnected terminal or Ctrl-C does not guarantee
remote cleanup. If creation returned no ID, the service might still have created
a session. Report that uncertainty; do not guess an ID, delete other sessions,
restart services, or retry until the organiser confirms it is safe. Temporary
forwarding closing is also not proof of remote session removal.

For missing files or credentials, host-key errors, HTTP/TLS errors, dataset
mismatch, readiness failures, statement errors, transfer failures or unconfirmed
cleanup: **stop and report** the command/activity, failing stage, error text,
printed session ID if available, assigned service, exit status, and relevant
output path. Never report private-key contents or credential payloads.

### Final reflection

- Trace one command from the terminal through forwarding, Livy, the driver,
  executor processing, HTTP output, and owned-session removal.
- Compare Step 1 and Step 2: what changed in transport, driver location, result
  retrieval and cleanup, and what remained the same in the analysis?
- Why are separate activity commands not one persistent interactive session?
- What do normal completion messages establish, and what must the organiser
  still inspect independently?

### Completion checklist

- [ ] I completed Step 1 and resolved its cleanup before Step 2 approval.
- [ ] I used the prepared terminal and ran one authorised command at a time.
- [ ] I recorded predictions, observations and explanations for all five activities.
- [ ] I distinguished session readiness from statement readiness.
- [ ] I compared the schema and counts with Step 1 for the same input snapshot.
- [ ] I checked activity 5's result separately from cleanup and final success.
- [ ] I saved my notebook answers and reported uncertainties before continuing.

<details>
<summary>Supplied-snapshot reference checks — consult after recording observations</summary>

| Check | Reference result |
|---|---:|
| Full input, activity 1 | 5,000 |
| After column selection, activity 2 | 5,000 |
| Age > 30, activity 3 | 4,376 |
| Personal Loan raw code 0, activity 4 | 4,520 |
| Personal Loan raw code 1, activity 4 | 480 |

These are reference checks from the existing supplied-snapshot walkthrough,
not evidence of a successful new participant run. Preview ordering may differ.
Report differences rather than editing data. Category meanings and redistribution
permissions require the dataset documentation and its stated limitations.

</details>

**Optional organiser-requested reference, not an extra required submission:**
`python3 02_spark_via_livy.py` runs the existing full analysis in one owned session.
The guided exercise uses the five separate activity commands instead.

Treat `docs/spark-DA.pdf` as conceptual/historical background, not permission to
run old deployment commands or notebook kernels. Publication preserves existing
participant files, so repository updates alone do not refresh old delivered copies.
This notebook adds no infrastructure changes or evidence of execution readiness.


### Your answers

**End-to-end Livy execution and result transfer**

_Write your answer here._

**Comparison with Step 1 and independent activity sessions**

_Write your answer here._

**Cleanup evidence, remaining limits and unresolved questions**

_Write your answer here._
